In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação baseada em variáveis globais explicadas pelo RF-Temp
- Sem RF-Comp residual
- Park apenas como COMPARAÇÃO opcional (fora do pipeline principal)

Pipeline:
1) Carrega dados, recorta 30–50 kHz, alinha colunas comuns
2) Define y_ref (mediana real @REF_TEMP, treino+prova se disponível)
3) Extrai features globais por curva
4) Treina RF-Temp para prever T a partir das features (interpretação da temperatura)
5) Ajusta modelos lineares feature ~ T (no treino) para estimar o "efeito da temperatura"
6) Compensa cada curva de teste para que (mean, amp, slope, ...) fiquem como em 20 °C
7) (Opcional) Aplica Park CLÁSSICO separadamente, só para COMPARAÇÃO (não usado pelo RF)
8) Reporta métricas e checagens com RF-Temp nas curvas compensadas
"""

import re, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics   import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression

# ===================== PARÂMETROS =====================
REF_TEMP      = 20
FREQ_MIN_KHZ  = 30
FREQ_MAX_KHZ  = 50
PKL_TREINO    = "base_treino.pkl"
PKL_PROVA     = "base_prova (1).pkl"

# Conjuntos fixos (pode alterar)
TEMPS_TREINO = {0, 10, 40, 60}
TEMPS_PROVA  = {-10, 30, 50, 70}

# RF-Temp (treinado sobre FEATURES, não sobre a curva inteira)
RF_TEMP_PARAMS = dict(
    n_estimators=600, max_depth=18,
    min_samples_leaf=2, min_samples_split=4,
    max_features="sqrt", bootstrap=True,
    n_jobs=-1, random_state=7
)

# Park (opcional) — apenas p/ comparação (NÃO entra no RF)
DO_PARK_COMPARE = True
MAX_SHIFT_IDX   = 40  # deslocamento máximo (em pontos de índice)
PARK_OVERLAP_MIN = 0.65  # fração mínima de overlap para cálculo de δS

# Caps suaves (p/ segurança ao aplicar transformações)
CAP_GAIN_FRAC   = 0.60   # ganho máximo relativo em torno da média
CAP_OFFSET_FRAC = 0.60   # offset máximo relativo à amplitude da curva
CAP_TILT_FRAC   = 0.60   # tilt máximo relativo (impacto nos extremos)

SMOOTH_WIN      = 7      # suavização final (ímpar); 1 = sem suavizar

# ===================== HELPERS =====================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f/1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def moving_average(arr, win):
    if win<=1 or win%2==0: return arr
    r=win//2
    padl = np.repeat(arr[:1], r)
    padr = np.repeat(arr[-1:], r)
    x = np.concatenate([padl, arr, padr])
    c = np.cumsum(x, dtype=float)
    c = np.concatenate([[0.0], c])
    s = c[win:] - c[:-win]
    return s/float(win)

# ===== métricas =====
def rmsd(y_ref, y):   return float(np.sqrt(np.mean((y_ref - y)**2)))
def ccdm(y_ref, y):
    y1, y2 = y_ref - y_ref.mean(), y - y.mean()
    den = (np.linalg.norm(y1)*np.linalg.norm(y2))+1e-12
    rho = float(np.clip(np.dot(y1, y2)/den, -1, 1))
    return 1.0 - rho
def corr_per_sample(Y, Yhat):
    out=[]
    for y,yh in zip(Y,Yhat):
        num=((y-y.mean())*(yh-yh.mean())).sum()
        den=np.sqrt(((y-y.mean())**2).sum()*((yh-yh.mean())**2).sum())+1e-12
        out.append(float(np.clip(num/den,-1,1)))
    return np.array(out)
def sam_per_sample(Y,Yhat):
    out=[]
    for y,yh in zip(Y,Yhat):
        den=(np.linalg.norm(y)*np.linalg.norm(yh))+1e-12
        cosang=float(np.clip(np.dot(y,yh)/den,-1,1))
        out.append(float(np.degrees(np.arccos(cosang))))
    return np.array(out)
def nrmse_per_sample(Y,Yhat):
    out=[]
    for y,yh in zip(Y,Yhat):
        rmse=np.sqrt(np.mean((y-yh)**2))
        rng=np.max(y)-np.min(y)
        out.append(float(rmse/(rng+1e-12)))
    return np.array(out)
def eval_all_metrics(Y_true, Y_pred):
    y1, y2 = Y_true.reshape(-1), Y_pred.reshape(-1)
    return dict(
        R2      = r2_score(y1, y2),
        RMSE    = float(np.sqrt(mean_squared_error(y1, y2))),
        MAE     = float(mean_absolute_error(y1, y2)),
        Corr    = float(corr_per_sample(Y_true, Y_pred).mean()),
        SAM_deg = float(sam_per_sample(Y_true, Y_pred).mean()),
        NRMSE   = float(nrmse_per_sample(Y_true, Y_pred).mean()),
        RMSD    = float(np.mean([rmsd(Y_true[i], Y_pred[i]) for i in range(Y_true.shape[0])])),
        CCDM    = float(np.mean([ccdm(Y_true[i], Y_pred[i]) for i in range(Y_true.shape[0])])),
    )
def print_metrics_block(title, m):
    print(f"\n== {title} ==")
    print(" | ".join([f"{k}={m[k]:.4f}" for k in ["R2","RMSE","MAE","Corr","SAM_deg","NRMSE","RMSD","CCDM"]]))

# ===================== FEATURES GLOBAIS =====================
# Escolhi variáveis interpretáveis e fáceis de inverter:
# - mean: nível (baseline)
# - amp: amplitude (max - min)
# - slope: inclinação global entre extremos
# - std: rugosidade geral
# - peak_pos_rel: posição relativa do pico principal (proxy de "shift")
def compute_global_features(X, fhz):
    n, m = X.shape
    x0 = X[:,0]; x1 = X[:,-1]
    mean  = X.mean(axis=1)
    std   = X.std(axis=1)
    amp   = X.max(axis=1) - X.min(axis=1)
    slope = (x1 - x0) / (fhz[-1] - fhz[0] + 1e-12)
    # posição do pico principal (relativa ao comprimento)
    peak_idx = np.argmax(X, axis=1)
    peak_pos_rel = peak_idx / (m-1 + 1e-12)
    F = np.column_stack([mean, std, amp, slope, peak_pos_rel])
    names = ["mean","std","amp","slope","peak_pos_rel"]
    return F, names

# ===================== LOAD =====================
base_tr = pd.read_pickle(PKL_TREINO)
base_te = pd.read_pickle(PKL_PROVA)

freq_cols_tr, _ = get_freq_columns(base_tr, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
freq_cols_te, _ = get_freq_columns(base_te, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
common_cols = [c for c in freq_cols_tr if c in freq_cols_te]
fhz = np.array([extract_freq_hz(c) for c in common_cols], float)
order = np.argsort(fhz); common_cols = [common_cols[i] for i in order]; fhz = fhz[order]
fkHz = fhz/1e3

# matrizes
X_tr_full = base_tr[common_cols].to_numpy(float)
X_te_full = base_te[common_cols].to_numpy(float)
T_tr_full = base_tr["temp_c"].to_numpy(float)
T_te_full = base_te["temp_c"].to_numpy(float)

# y_ref real @20 °C (mediana treino+prova se houver)
pool_20=[]
if (base_tr["temp_c"]==REF_TEMP).any():
    pool_20.append(base_tr.loc[base_tr["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
if (base_te["temp_c"]==REF_TEMP).any():
    pool_20.append(base_te.loc[base_te["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
assert len(pool_20)>0, "Não há curva real @20°C!"
y_ref = np.median(np.vstack(pool_20), axis=0)

# Conjuntos fixos
tr_restr = base_tr[base_tr["temp_c"].isin(TEMPS_TREINO)].copy()
te_restr = base_te[base_te["temp_c"].isin(TEMPS_PROVA)].copy()

X_tr = tr_restr[common_cols].to_numpy(float)
X_te = te_restr[common_cols].to_numpy(float)
T_tr = tr_restr["temp_c"].to_numpy(float)
T_te = te_restr["temp_c"].to_numpy(float)

# ===================== RF-TEMP EM FEATURES =====================
F_tr, feat_names = compute_global_features(X_tr, fhz)
F_te, _          = compute_global_features(X_te, fhz)

rf_temp = RandomForestRegressor(**RF_TEMP_PARAMS).fit(F_tr, T_tr)

print("\n== RF-Temp (features globais) ==")
print(f"R²(treino) = {rf_temp.score(F_tr, T_tr):.3f}")
T_te_pred_from_feats = rf_temp.predict(F_te)
print(f"Pred T(te) — média={T_te_pred_from_feats.mean():.2f} °C | desvio={T_te_pred_from_feats.std():.2f} °C")

print("\nImportâncias das features (RF-Temp):")
for name, imp in sorted(zip(feat_names, rf_temp.feature_importances_), key=lambda x: -x[1]):
    print(f"  {name:>12s}: {imp:.3f}")

# ===================== MODELOS FEATURE ~ TEMPERATURA (LINEAR) =====================
# Ajusta, no TREINO, regressões lineares por feature: f = a*T + b
# Depois usamos isso p/ estimar f_20 (alvo) e inverter no domínio da curva (mean/amp/slope)
def fit_feature_vs_temp_models(F, T, feat_names):
    models = {}
    T = np.asarray(T).reshape(-1,1)
    for j, name in enumerate(feat_names):
        lr = LinearRegression().fit(T, F[:,j])
        models[name] = lr
    return models

feat_models = fit_feature_vs_temp_models(F_tr, T_tr, feat_names)

def feature_targets_at_ref(feat_models, ref_temp=REF_TEMP):
    targets = {}
    Tref = np.array([[ref_temp]])
    for name, lr in feat_models.items():
        targets[name] = float(lr.predict(Tref)[0])
    return targets

feat_targets_20 = feature_targets_at_ref(feat_models, REF_TEMP)

print("\nAlvos de features para 20 °C (do modelo linear no treino):")
for k,v in feat_targets_20.items():
    print(f"  {k:>12s}: {v:.6f}")

# ===================== COMPENSAÇÃO NO DOMÍNIO DA CURVA (mean, amp, slope) =====================
# Aplicamos correções na curva para que suas features se aproximem dos alvos a 20 °C.
# - offset p/ média
# - ganho p/ amplitude (em torno da média)
# - tilt p/ slope (aplica rampa linear ao longo de f)
def apply_compensation_by_features(x, fhz, targets, caps):
    x = x.copy()
    mean_t  = targets["mean"]
    amp_t   = targets["amp"]
    slope_t = targets["slope"]

    # features atuais
    mean_x  = float(x.mean())
    amp_x   = float(x.max() - x.min())
    slope_x = float((x[-1] - x[0]) / (fhz[-1] - fhz[0] + 1e-12))

    # 1) Offset (muda a média)
    offset = (mean_t - mean_x)

    # CAP de offset (relativo à amplitude da curva)
    offset_cap = caps["offset_frac"] * max(1e-9, amp_x)
    offset = float(np.clip(offset, -offset_cap, offset_cap))

    x = x + offset

    # 2) Ganho (ajusta amplitude em torno da média)
    # alvo de ganho = amp_t / amp_x  (se amp_x ~= 0, ganho=1)
    gain = 1.0
    if amp_x > 1e-9:
        gain = float(amp_t / amp_x)
    # CAP de gain relativo: limita distância de 1.0
    g_min = 1.0 - caps["gain_frac"]
    g_max = 1.0 + caps["gain_frac"]
    gain = float(np.clip(gain, g_min, g_max))

    x = mean_t + gain*(x - mean_t)

    # 3) Tilt (inclinação): queremos (x[-1]-x[0])/(Δf) ≈ slope_t
    # Δslope desejado:
    delta_slope = slope_t - slope_x
    # Construir rampa linear que varia de -0.5 a +0.5 ao longo do espectro
    u = np.linspace(-0.5, 0.5, len(x))
    df = (fhz[-1] - fhz[0] + 1e-12)
    tilt_signal = (delta_slope * df) * u

    # CAP do tilt: limita o quanto adicionamos nos extremos (relativo à amp)
    tilt_cap = caps["tilt_frac"] * max(1e-9, amp_x)
    tilt_signal = np.clip(tilt_signal, -tilt_cap, tilt_cap)

    x = x + tilt_signal

    return x

def compensate_set_by_features(X, fhz, feat_models, ref_temp=REF_TEMP,
                               cap_gain_frac=CAP_GAIN_FRAC,
                               cap_offset_frac=CAP_OFFSET_FRAC,
                               cap_tilt_frac=CAP_TILT_FRAC,
                               smooth_win=SMOOTH_WIN):
    targets = feature_targets_at_ref(feat_models, ref_temp)
    Y = np.zeros_like(X)
    caps = dict(gain_frac=cap_gain_frac, offset_frac=cap_offset_frac, tilt_frac=cap_tilt_frac)
    for i in range(X.shape[0]):
        yi = apply_compensation_by_features(X[i], fhz, targets, caps)
        if smooth_win>1 and (smooth_win%2==1):
            yi = moving_average(yi, smooth_win)
        Y[i] = yi
    return Y, targets

# === Aplica na PROVA ===
t0 = time.time()
Y_te_hat, targets_used = compensate_set_by_features(X_te, fhz, feat_models, REF_TEMP)
print(f"\n[INFO] Compensação por features aplicada em {time.time()-t0:.2f}s")

# ===================== MÉTRICAS VS REFERÊNCIA E VS ORIGINAL =====================
Y_ref_te = np.tile(y_ref, (X_te.shape[0],1))

print("\n### MÉTRICAS — FINAL ###")
m_vs_ref  = eval_all_metrics(Y_ref_te, Y_te_hat); print_metrics_block("FINAL vs REF",  m_vs_ref)
m_vs_orig = eval_all_metrics(X_te,     Y_te_hat); print_metrics_block("FINAL vs ORIG", m_vs_orig)

# Checagem RF-Temp nas curvas finais (deveria tender a ~20 °C)
F_te_final, _ = compute_global_features(Y_te_hat, fhz)
T_hat_final = rf_temp.predict(F_te_final)
print("\n### Checagem RF-Temp nas curvas finais ###")
print(f"média={float(np.mean(T_hat_final)):.2f}°C | desvio={float(np.std(T_hat_final)):.2f}°C | MAE vs {REF_TEMP}°C={float(np.mean(np.abs(T_hat_final-REF_TEMP))):.2f}°C")

# ===================== (OPCIONAL) PARK CLÁSSICO — APENAS COMPARAÇÃO =====================
def park_classic_single(x, y_ref, max_shift_idx=MAX_SHIFT_IDX, overlap_min=PARK_OVERLAP_MIN):
    """
    Park clássico:
      - varre deslocamento inteiro k em índice ([-max_shift, +max_shift])
      - calcula δS = média(Y_ref - X_shift) no overlap
      - escolhe (k, δS) que minimiza a variância do residual
    Retorna a curva x alinhada por Park.
    """
    n = len(x)
    best = (np.inf, 0, 0.0)  # (mse, k, delta)
    for k in range(-max_shift_idx, max_shift_idx+1):
        if k < 0:
            xs = x[-k:n]
            yr = y_ref[0:n+k]
        elif k > 0:
            xs = x[0:n-k]
            yr = y_ref[k:n]
        else:
            xs = x
            yr = y_ref
        if len(xs) < int(overlap_min*n):  # exige overlap mínimo
            continue
        delta = float(np.mean(yr - xs))
        resid = yr - (xs + delta)
        mse = float(np.mean(resid**2))
        if mse < best[0]:
            best = (mse, k, delta)

    _, kbest, dbest = best
    # reconstrói curva final de mesmo tamanho n:
    if kbest < 0:
        xs = x[-kbest:n] + dbest
        yout = np.empty_like(x); yout[:] = np.nan
        yout[0:n+kbest] = xs
        # preenche bordas mantendo extremos
        yout[n+kbest:] = (x[-1] + dbest)
    elif kbest > 0:
        xs = x[0:n-kbest] + dbest
        yout = np.empty_like(x); yout[:] = np.nan
        yout[kbest:n] = xs
        yout[:kbest] = (x[0] + dbest)
    else:
        yout = x + dbest

    # simples suavização das “costuras”
    yout = moving_average(yout, 5)
    return yout

if DO_PARK_COMPARE:
    print("\n[COMPARAÇÃO] Executando Park clássico nas curvas de teste (fora do pipeline principal)...")
    Y_te_park = np.zeros_like(X_te)
    for i in range(X_te.shape[0]):
        Y_te_park[i] = park_classic_single(X_te[i], y_ref)
    m_park_vs_ref  = eval_all_metrics(Y_ref_te, Y_te_park); print_metrics_block("PARK vs REF",  m_park_vs_ref)
    m_park_vs_orig = eval_all_metrics(X_te,     Y_te_park); print_metrics_block("PARK vs ORIG", m_park_vs_orig)

# ===================== PLOT =====================
def _prep_plot():
    plt.rcParams.update({
        "figure.figsize": (8.8, 4.8),
        "axes.grid": True, "grid.alpha": 0.28,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.labelsize": 12, "axes.titlesize": 13,
        "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 10, "lines.linewidth": 1.8,
    })

def plot_comp(i=0, save=False, prefix="feat_comp"):
    _prep_plot()
    fhz_khz = fkHz
    T_real  = float(te_restr.iloc[i]["temp_c"])

    # RF-Temp “percebido” antes e depois
    feats_orig, _  = compute_global_features(X_te[i][None,:], fhz)
    feats_final, _ = compute_global_features(Y_te_hat[i][None,:], fhz)
    T_pred_orig  = float(rf_temp.predict(feats_orig)[0])
    T_pred_final = float(rf_temp.predict(feats_final)[0])

    fig, ax = plt.subplots()
    ax.plot(fhz_khz, X_te[i],    label=f"Original @ {T_real:.0f} °C (RF-Temp≈{T_pred_orig:.1f} °C)",  color="#1f77b4")
    ax.plot(fhz_khz, y_ref,      label=f"Referência @ {REF_TEMP} °C", color="#ff7f0e")
    ax.plot(fhz_khz, Y_te_hat[i],label=f"Compensada (RF-Temp≈{T_pred_final:.1f} °C)", color="#2ca02c")
    ax.set_title(f"Amostra {i} — Compensação por variáveis (mean/amp/slope)")
    ax.set_xlabel("Frequência (kHz)"); ax.set_ylabel("Re{Z}")
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    if save: fig.savefig(f"{prefix}_i{i}.png", dpi=300)
    plt.show()

# Exemplo
plot_comp(i=0, save=False)
